# **1. INSTALL LIBRARY**

In [15]:
# Streamlit → UI chatbot / web app
!pip install -q streamlit

# Gemini SDK → komunikasi dengan Gemini API (AI Model)
!pip install -q google-genai

# Embedding + similarity → semantic search / pencarian context mirip
!pip install -q sentence-transformers scikit-learn

# Vector database → menyimpan embedding untuk memory / RAG
!pip install -q chromadb

# Membuat public URL dari Colab → agar Streamlit bisa dibuka di browser
!pip install -q pyngrok

## **2. IMPORT LIBRARY & SETUP DASAR**

In [3]:
# =========================================
# IMPORT LIBRARY
# =========================================

import os
import json
import uuid

import streamlit as st

# Gemini API
from google import genai

# Embedding Model
from sentence_transformers import SentenceTransformer

# Similarity
from sklearn.metrics.pairwise import cosine_similarity

# Vector Database
import chromadb

# =========================================
# LOAD EMBEDDING MODEL
# =========================================

@st.cache_resource
def load_embedding_model():
    return SentenceTransformer('all-MiniLM-L6-v2')

embedding_model = load_embedding_model()

print("✅ Embedding model loaded")


# =========================================
# CREATE VECTOR DATABASE
# =========================================

chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="campus_helpdesk_memory"
)

print("✅ ChromaDB initialized")


# =========================================
# USER MEMORY
# =========================================

user_memory = {}

print("✅ User memory initialized")


# =========================================
# TOXIC WORD LIST
# =========================================

TOXIC_WORDS = [
    "goblok",
    "tolol",
    "anjing",
    "bodoh",
    "bangsat",
    "kampret",
    "tai",
    "kontol",
    "memek"
]

print("✅ Toxic filter initialized")


# =========================================
# SYSTEM STATUS
# =========================================

print("\n🚀 Campus Helpdesk (Chatbot) telah dikonfigurasi.")

2026-05-27 00:46:37.572 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded
✅ ChromaDB initialized
✅ User memory initialized
✅ Toxic filter initialized

🚀 Campus Helpdesk (Chatbot) telah dikonfigurasi.


# **3. SETUP GEMINI API KEY**

In [4]:
from google.colab import userdata

# =========================================
# LOAD SECRET FROM COLAB
# =========================================
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI")

# =========================================
# LOAD API KEY FROM ENVIRONMENT VARIABLE
# =========================================

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# =========================================
# GEMINI CLIENT
# =========================================

client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("✅ Gemini API berhasil dikoneksikan secara aman.")

✅ Gemini API berhasil dikoneksikan secara aman.


# **4. DATASET KAMPUS (KNOWLEDGE BASE)**

In [5]:
# =========================================
# DATASET KAMPUS (KNOWLEDGE BASE)
# =========================================

campus_data = [

    # =================================================
    # UNIVERSITAS
    # =================================================

    {
        "category": "universitas",
        "title": "Universitas Teknologi Nusantara",
        "keywords": [
            "universitas",
            "kampus terbaik",
            "kampus teknologi"
        ],
        "content": """
        Universitas Teknologi Nusantara merupakan
        universitas berbasis teknologi dan AI.

        Universitas memiliki:
        - Fakultas Teknik
        - Fakultas Bisnis
        - Fakultas Desain

        Akreditasi Universitas: Unggul
        Lokasi: Surabaya
        """
    },

    # =================================================
    # PROGRAM STUDI
    # =================================================

    {
        "category": "program_studi",
        "title": "Teknik Informatika",
        "keywords": [
            "informatika",
            "AI",
            "programming",
            "coding"
        ],
        "content": """
        Program Studi Teknik Informatika fokus pada:
        - Artificial Intelligence
        - Machine Learning
        - Data Science
        - Web Development
        - Mobile Development
        - Cyber Security

        Akreditasi: A
        Jenjang: S1

        Prospek karier:
        - Software Engineer
        - AI Engineer
        - Data Scientist
        - Backend Developer
        """
    },

    {
        "category": "program_studi",
        "title": "Sistem Informasi",
        "keywords": [
            "sistem informasi",
            "business",
            "data analytics"
        ],
        "content": """
        Program Studi Sistem Informasi fokus pada:
        - Business Intelligence
        - ERP
        - UI/UX
        - Data Analytics
        - Project Management

        Akreditasi: A
        Jenjang: S1

        Prospek karier:
        - System Analyst
        - IT Consultant
        - Data Analyst
        """
    },

    # =================================================
    # FAKULTAS
    # =================================================

    {
        "category": "fakultas",
        "title": "Fakultas Teknik",
        "keywords": [
            "fakultas",
            "teknik"
        ],
        "content": """
        Fakultas Teknik menaungi:
        - Teknik Informatika
        - Sistem Informasi
        - Teknik Industri

        Fakultas memiliki:
        - Laboratorium AI
        - Laboratorium Komputer
        - Ruang Multimedia
        """
    },

    # =================================================
    # UKT / BIAYA
    # =================================================

    {
        "category": "ukt",
        "title": "UKT Teknik Informatika",
        "keywords": [
            "ukt",
            "biaya",
            "informatika"
        ],
        "content": """
        UKT Teknik Informatika berkisar:
        - Rp 5.000.000
        - Rp 7.500.000

        per semester tergantung jalur masuk.
        """
    },

    {
        "category": "ukt",
        "title": "UKT Sistem Informasi",
        "keywords": [
            "ukt",
            "biaya",
            "sistem informasi"
        ],
        "content": """
        UKT Sistem Informasi berkisar:
        - Rp 4.500.000
        - Rp 6.500.000

        per semester tergantung jalur masuk.
        """
    },

    # =================================================
    # BEASISWA
    # =================================================

    {
        "category": "beasiswa",
        "title": "Beasiswa Prestasi Akademik",
        "keywords": [
            "beasiswa",
            "prestasi",
            "ipk"
        ],
        "content": """
        Beasiswa akademik diberikan kepada mahasiswa
        dengan IPK minimal 3.5.

        Benefit:
        - Potongan UKT 50%
        - Bantuan penelitian
        - Sertifikat prestasi
        """
    },

    {
        "category": "beasiswa",
        "title": "Beasiswa Organisasi",
        "keywords": [
            "beasiswa organisasi",
            "organisasi"
        ],
        "content": """
        Beasiswa organisasi diberikan kepada mahasiswa
        aktif organisasi kampus.

        Benefit:
        - Potongan UKT
        - Dana kegiatan mahasiswa
        """
    },

    # =================================================
    # FASILITAS
    # =================================================

    {
        "category": "fasilitas",
        "title": "Fasilitas Kampus",
        "keywords": [
            "fasilitas",
            "laboratorium",
            "wifi"
        ],
        "content": """
        Kampus memiliki fasilitas:
        - WiFi kampus
        - Laboratorium komputer
        - Perpustakaan digital
        - Studio multimedia
        - Coworking space
        - Auditorium
        - Kantin mahasiswa
        """
    },

    # =================================================
    # AKREDITASI
    # =================================================

    {
        "category": "akreditasi",
        "title": "Akreditasi Kampus",
        "keywords": [
            "akreditasi",
            "unggul"
        ],
        "content": """
        Universitas Teknologi Nusantara memiliki:
        - Akreditasi Universitas: Unggul

        Program Studi:
        - Teknik Informatika: A
        - Sistem Informasi: A
        """
    },

    # =================================================
    # KARIER
    # =================================================

    {
        "category": "karier",
        "title": "Prospek Karier Lulusan",
        "keywords": [
            "karier",
            "pekerjaan",
            "lulusan"
        ],
        "content": """
        Lulusan kampus memiliki prospek karier:
        - Software Engineer
        - AI Engineer
        - Data Scientist
        - UI/UX Designer
        - System Analyst
        - IT Consultant
        """
    },

    # =================================================
    # PENDAFTARAN
    # =================================================

    {
        "category": "pendaftaran",
        "title": "Pendaftaran Mahasiswa Baru",
        "keywords": [
            "pendaftaran",
            "daftar kuliah",
            "mahasiswa baru"
        ],
        "content": """
        Persyaratan pendaftaran:
        - Fotokopi ijazah
        - KTP
        - Pas foto
        - Mengisi formulir online

        Jalur pendaftaran:
        - Reguler
        - Prestasi
        - Beasiswa
        """
    },

    # =================================================
    # KURIKULUM
    # =================================================

    {
        "category": "kurikulum",
        "title": "Kurikulum Teknik Informatika",
        "keywords": [
            "kurikulum",
            "mata kuliah",
            "informatika"
        ],
        "content": """
        Mata kuliah unggulan:
        - Artificial Intelligence
        - Machine Learning
        - Deep Learning
        - Basis Data
        - Pemrograman Web
        - Mobile Development
        """
    },

    # =================================================
    # LOKASI
    # =================================================

    {
        "category": "lokasi",
        "title": "Lokasi Kampus",
        "keywords": [
            "lokasi",
            "alamat",
            "surabaya"
        ],
        "content": """
        Kampus berlokasi di:
        Surabaya, Jawa Timur.

        Akses lokasi:
        - Dekat terminal
        - Dekat stasiun
        - Area pusat pendidikan
        """
    }

]

print("✅ Dataset kampus berhasil dibuat")


# =========================================
# MASUKKAN KE VECTOR DATABASE
# =========================================

if collection.count() == 0:

    for data in campus_data:

        embedding = embedding_model.encode(
            data["content"]
        ).tolist()

        collection.add(
            ids=[str(uuid.uuid4())],
            embeddings=[embedding],
            documents=[data["content"]],
            metadatas=[{
                "category": data["category"],
                "title": data["title"],
                "keywords": ", ".join(data["keywords"])
            }]
        )

    print("✅ Data masuk ke ChromaDB")

else:

    print("✅ ChromaDB sudah memiliki data")


# =========================================
# CEK TOTAL DATA
# =========================================

print("\n📚 TOTAL DATA KNOWLEDGE BASE:")
print(len(campus_data))

✅ Dataset kampus berhasil dibuat
✅ Data masuk ke ChromaDB

📚 TOTAL DATA KNOWLEDGE BASE:
14


# **5. TOXIC FILTER SYSTEM**

In [6]:
# =========================================
# TOXIC FILTER SYSTEM
# =========================================

MAX_TOXIC_SCORE = 3


# =========================================
# INITIALIZE USER MEMORY
# =========================================

def initialize_user(user_id):

    if user_id not in user_memory:

        user_memory[user_id] = {
            "history": [],
            "topics": [],
            "repeated_questions": [],
            "toxic_score": 0,
            "warnings": []
        }


# =========================================
# DETECT TOXIC MESSAGE
# =========================================

def detect_toxicity(text):

    text = text.lower()

    detected_words = []

    for word in TOXIC_WORDS:

        if word in text:

            detected_words.append(word)

    return detected_words


# =========================================
# HANDLE TOXIC USER
# =========================================

def handle_toxicity(user_id, toxic_words):

    user_memory[user_id]["toxic_score"] += 1

    toxic_score = user_memory[user_id]["toxic_score"]

    # =====================================
    # SAVE WARNING HISTORY
    # =====================================

    user_memory[user_id]["warnings"].append({
        "toxic_words": toxic_words,
        "level": toxic_score
    })

    # =====================================
    # LEVEL 1
    # =====================================

    if toxic_score == 1:

        return """
⚠️ Mohon gunakan bahasa yang sopan saat menggunakan layanan Campus Helpdesk.

Saya tetap siap membantu Anda dalam menemukan informasi terkait:
- Universitas
- Jurusan
- UKT
- Beasiswa
- Kurikulum
- Fasilitas kampus
"""

    # =====================================
    # LEVEL 2
    # =====================================

    elif toxic_score == 2:

        return """
⚠️ Terdeteksi penggunaan bahasa yang tidak sesuai.

Agar percakapan tetap nyaman dan profesional,
mohon gunakan bahasa yang lebih baik.

Silakan lanjutkan pertanyaan terkait kebutuhan akademik atau informasi kampus.
"""

    # =====================================
    # LEVEL 3 (BLOCK USER)
    # =====================================

    elif toxic_score >= MAX_TOXIC_SCORE:

        return """
⛔ Percakapan sementara dihentikan karena terdeteksi penggunaan bahasa yang melanggar etika layanan kampus.

Silakan coba kembali nanti dengan bahasa yang lebih sopan dan konstruktif.
"""


# =========================================
# CHECK USER BLOCKED
# =========================================

def is_user_blocked(user_id):

    return (
        user_memory[user_id]["toxic_score"]
        >= MAX_TOXIC_SCORE
    )


# =========================================
# RESET TOXIC SCORE
# =========================================

def reset_toxic_score(user_id):

    user_memory[user_id]["toxic_score"] = 0

    user_memory[user_id]["warnings"] = []


# =========================================
# SHOW USER MODERATION STATUS
# =========================================

def show_moderation_status(user_id):

    print("\n🛡️ MODERATION STATUS")

    print(
        f"Toxic Score: "
        f"{user_memory[user_id]['toxic_score']}"
    )

    print(
        f"Total Warnings: "
        f"{len(user_memory[user_id]['warnings'])}"
    )


# =========================================
# TEST TOXIC FILTER
# =========================================

test_user = "demo_user"

initialize_user(test_user)

sample_text = "kampus ini goblok"

detected_words = detect_toxicity(sample_text)

if detected_words:

    response = handle_toxicity(
        test_user,
        detected_words
    )

    print("🚨 Toxic Message Detected\n")

    print("Detected Words:")
    print(detected_words)

    print("\nResponse:")
    print(response)

else:

    print("✅ Safe Message")


# =========================================
# SHOW STATUS
# =========================================

show_moderation_status(test_user)

🚨 Toxic Message Detected

Detected Words:
['goblok']

Response:

⚠️ Mohon gunakan bahasa yang sopan saat menggunakan layanan Campus Helpdesk.

Saya tetap siap membantu Anda dalam menemukan informasi terkait:
- Universitas
- Jurusan
- UKT
- Beasiswa
- Kurikulum
- Fasilitas kampus


🛡️ MODERATION STATUS
Toxic Score: 1
Total Warnings: 1


# **6. MEMORY & REPEATED QUESTION SYSTEM**

In [7]:
# =========================================
# MEMORY & REPEATED QUESTION SYSTEM
# =========================================

SIMILARITY_THRESHOLD = 0.90

MAX_HISTORY = 20


# =========================================
# SAVE CHAT HISTORY
# =========================================

def save_to_memory(user_id, question):

    # =====================================
    # LIMIT CHAT HISTORY
    # =====================================

    if len(user_memory[user_id]["history"]) >= MAX_HISTORY:

        user_memory[user_id]["history"].pop(0)

    # =====================================
    # SAVE QUESTION
    # =====================================

    user_memory[user_id]["history"].append(question)


# =========================================
# GET QUESTION EMBEDDING
# =========================================

def get_embedding(text):

    return embedding_model.encode(text)


# =========================================
# DETECT REPEATED QUESTION
# =========================================

def is_repeated_question(user_id, new_question):

    history = user_memory[user_id]["history"]

    # =====================================
    # EMPTY HISTORY
    # =====================================

    if len(history) == 0:

        return False, None

    # =====================================
    # NEW QUESTION EMBEDDING
    # =====================================

    new_embedding = get_embedding(
        new_question
    )

    best_similarity = 0

    most_similar_question = None

    # =====================================
    # CHECK ALL HISTORY
    # =====================================

    for old_question in history:

        old_embedding = get_embedding(
            old_question
        )

        similarity = cosine_similarity(
            [new_embedding],
            [old_embedding]
        )[0][0]

        # =================================
        # SAVE BEST MATCH
        # =================================

        if similarity > best_similarity:

            best_similarity = similarity

            most_similar_question = old_question

    # =====================================
    # SIMILAR QUESTION DETECTED
    # =====================================

    if best_similarity >= SIMILARITY_THRESHOLD:

        user_memory[user_id][
            "repeated_questions"
        ].append({
            "question": new_question,
            "similar_to": most_similar_question,
            "similarity_score": float(best_similarity)
        })

        return True, best_similarity

    return False, best_similarity


# =========================================
# REPEATED QUESTION RESPONSE
# =========================================

def repeated_question_response(similarity_score):

    similarity_percent = round(
        similarity_score * 100,
        1
    )

    return f"""
🔁 Pertanyaan serupa terdeteksi.

Kemiripan pertanyaan:
{similarity_percent}%

Silakan lihat jawaban sebelumnya terlebih dahulu agar informasi tidak berulang.

Jika ingin informasi lebih detail atau berbeda,
silakan ajukan pertanyaan yang lebih spesifik.
"""


# =========================================
# TRACK USER INTEREST
# =========================================

def track_user_interest(user_id, text):

    text = text.lower()

    keywords = [

        # =================================
        # PROGRAM STUDI
        # =================================

        "informatika",
        "sistem informasi",
        "data science",
        "cyber security",
        "ai",

        # =================================
        # KAMPUS
        # =================================

        "universitas",
        "fakultas",
        "kampus",

        # =================================
        # AKADEMIK
        # =================================

        "kurikulum",
        "mata kuliah",
        "akreditasi",

        # =================================
        # ADMINISTRASI
        # =================================

        "biaya",
        "ukt",
        "beasiswa",
        "pendaftaran",

        # =================================
        # FASILITAS
        # =================================

        "laboratorium",
        "wifi",
        "perpustakaan",

        # =================================
        # KARIER
        # =================================

        "karier",
        "prospek kerja",
        "pekerjaan"
    ]

    # =====================================
    # SAVE USER INTEREST
    # =====================================

    for keyword in keywords:

        if keyword in text:

            if keyword not in user_memory[user_id]["topics"]:

                user_memory[user_id]["topics"].append(
                    keyword
                )


# =========================================
# GET USER FAVORITE TOPICS
# =========================================

def get_user_topics(user_id):

    return user_memory[user_id]["topics"]


# =========================================
# SHOW USER PROFILE
# =========================================

def show_user_profile(user_id):

    profile = user_memory[user_id]

    print("\n👤 USER PROFILE")

    print(f"""
History Count       : {len(profile['history'])}
Topics Interested   : {profile['topics']}
Repeated Questions  : {len(profile['repeated_questions'])}
Toxic Score         : {profile['toxic_score']}
""")


# =========================================
# CLEAR USER MEMORY
# =========================================

def clear_user_memory(user_id):

    user_memory[user_id] = {
        "history": [],
        "topics": [],
        "repeated_questions": [],
        "toxic_score": 0,
        "warnings": []
    }


# =========================================
# TEST MEMORY SYSTEM
# =========================================

test_user = "mahasiswa_001"

initialize_user(test_user)

question_1 = (
    "berapa biaya kuliah teknik informatika?"
)

question_2 = (
    "biaya teknik informatika berapa ya?"
)

question_3 = (
    "apakah ada beasiswa kampus?"
)

# =========================================
# SAVE FIRST QUESTION
# =========================================

save_to_memory(
    test_user,
    question_1
)

# =========================================
# CHECK REPEATED QUESTION
# =========================================

is_repeat, similarity_score = (
    is_repeated_question(
        test_user,
        question_2
    )
)

print("\n🔍 REPEATED QUESTION CHECK")

print("Is Repeated:", is_repeat)

if is_repeat:

    print(
        repeated_question_response(
            similarity_score
        )
    )

# =========================================
# TRACK USER INTEREST
# =========================================

track_user_interest(
    test_user,
    question_1
)

track_user_interest(
    test_user,
    question_3
)

# =========================================
# SHOW PROFILE
# =========================================

show_user_profile(test_user)


🔍 REPEATED QUESTION CHECK
Is Repeated: True

🔁 Pertanyaan serupa terdeteksi.

Kemiripan pertanyaan:
94.19999694824219%

Silakan lihat jawaban sebelumnya terlebih dahulu agar informasi tidak berulang.

Jika ingin informasi lebih detail atau berbeda,
silakan ajukan pertanyaan yang lebih spesifik.


👤 USER PROFILE

History Count       : 1
Topics Interested   : ['informatika', 'biaya', 'kampus', 'beasiswa']
Repeated Questions  : 1
Toxic Score         : 0



# **7. SEMANTIC SEARCH + RAG SYSTEM**

In [8]:
# =========================================
# SEMANTIC SEARCH + RAG SYSTEM
# =========================================

TOP_K_RESULTS = 3

MIN_RELEVANCE_SCORE = 0.45


# =========================================
# SEARCH RELEVANT KNOWLEDGE
# =========================================

def search_knowledge(user_question):

    # =====================================
    # QUESTION EMBEDDING
    # =====================================

    question_embedding = embedding_model.encode(
        user_question
    ).tolist()

    # =====================================
    # SEARCH VECTOR DATABASE
    # =====================================

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=TOP_K_RESULTS
    )

    documents = results["documents"][0]

    metadatas = results["metadatas"][0]

    distances = results["distances"][0]

    knowledge = []

    # =====================================
    # PROCESS SEARCH RESULTS
    # =====================================

    for doc, meta, distance in zip(
        documents,
        metadatas,
        distances
    ):

        # =================================
        # CONVERT DISTANCE TO SCORE
        # =================================

        relevance_score = 1 - distance

        # =================================
        # SKIP LOW RELEVANCE
        # =================================

        if relevance_score < MIN_RELEVANCE_SCORE:

            continue

        knowledge.append({

            "title": meta["title"],

            "category": meta["category"],

            "keywords": meta.get(
                "keywords",
                ""
            ),

            "content": doc,

            "relevance_score": round(
                relevance_score,
                3
            )
        })

    return knowledge


# =========================================
# BUILD RAG CONTEXT
# =========================================

def build_context(knowledge):

    # =====================================
    # EMPTY KNOWLEDGE
    # =====================================

    if len(knowledge) == 0:

        return """
Tidak ditemukan informasi yang relevan
di knowledge base kampus.
"""

    context = ""

    # =====================================
    # BUILD CONTEXT
    # =====================================

    for index, item in enumerate(
        knowledge,
        start=1
    ):

        context += f"""
==============================
DATA {index}
==============================

Kategori:
{item['category']}

Judul:
{item['title']}

Keywords:
{item['keywords']}

Relevance Score:
{item['relevance_score']}

Isi Informasi:
{item['content']}

"""

    return context


# =========================================
# CREATE FINAL RAG PROMPT
# =========================================

def build_rag_prompt(user_question):

    # =====================================
    # SEARCH KNOWLEDGE
    # =====================================

    knowledge = search_knowledge(
        user_question
    )

    # =====================================
    # BUILD CONTEXT
    # =====================================

    context = build_context(
        knowledge
    )

    # =====================================
    # FINAL PROMPT
    # =====================================

    final_prompt = f"""
Kamu adalah AI Campus Helpdesk
yang membantu mahasiswa mencari
informasi akademik dan kampus.

Jawablah berdasarkan knowledge base berikut.

========================================
KNOWLEDGE BASE
========================================

{context}

========================================
ATURAN MENJAWAB
========================================

- Jawab dengan sopan dan profesional
- Gunakan bahasa Indonesia
- Fokus pada informasi kampus
- Jangan membuat informasi palsu
- Jika informasi tidak tersedia,
  katakan dengan jujur
- Buat jawaban jelas dan terstruktur

========================================
PERTANYAAN USER
========================================

{user_question}

========================================
JAWABAN AI
========================================
"""

    return final_prompt


# =========================================
# SHOW SEARCH RESULT
# =========================================

def show_search_result(knowledge):

    print("\n📚 SEARCH RESULT\n")

    if len(knowledge) == 0:

        print("Tidak ada data relevan")

        return

    for index, item in enumerate(
        knowledge,
        start=1
    ):

        print(
            f"""
================================
RESULT {index}
================================

Kategori :
{item['category']}

Judul :
{item['title']}

Relevance :
{item['relevance_score']}

Isi :
{item['content']}
"""
        )


# =========================================
# TEST RAG SYSTEM
# =========================================

question = (
    "berapa biaya kuliah teknik informatika?"
)

# =========================================
# SEARCH KNOWLEDGE
# =========================================

knowledge_result = search_knowledge(
    question
)

# =========================================
# SHOW RESULT
# =========================================

show_search_result(
    knowledge_result
)

# =========================================
# BUILD FINAL PROMPT
# =========================================

final_prompt = build_rag_prompt(
    question
)

# =========================================
# PREVIEW PROMPT
# =========================================

print("\n🧠 FINAL RAG PROMPT\n")

print(final_prompt[:2500])


📚 SEARCH RESULT

Tidak ada data relevan

🧠 FINAL RAG PROMPT


Kamu adalah AI Campus Helpdesk
yang membantu mahasiswa mencari
informasi akademik dan kampus.

Jawablah berdasarkan knowledge base berikut.

KNOWLEDGE BASE


Tidak ditemukan informasi yang relevan
di knowledge base kampus.


ATURAN MENJAWAB

- Jawab dengan sopan dan profesional
- Gunakan bahasa Indonesia
- Fokus pada informasi kampus
- Jangan membuat informasi palsu
- Jika informasi tidak tersedia,
  katakan dengan jujur
- Buat jawaban jelas dan terstruktur

PERTANYAAN USER

berapa biaya kuliah teknik informatika?

JAWABAN AI



# **8. SYSTEM PROMPT + AI RESPONSE ENGINE**

In [9]:
# =========================================
# SYSTEM PROMPT + AI RESPONSE ENGINE
# =========================================

SYSTEM_PROMPT = """
Kamu adalah AI Campus Helpdesk modern di Indonesia.

Tugas utama kamu:
- Membantu mahasiswa dan calon mahasiswa
- Menjawab pertanyaan akademik
- Menjawab administrasi kampus
- Memberikan informasi universitas
- Memberikan informasi jurusan
- Memberikan informasi UKT
- Memberikan informasi beasiswa
- Memberikan informasi fasilitas kampus
- Memberikan informasi akreditasi
- Memberikan informasi karier lulusan

Aturan penting:
- Gunakan bahasa Indonesia
- Gunakan bahasa formal dan profesional
- Bersikap seperti staf akademik universitas
- Jangan menggunakan bahasa kasar
- Jangan membuat informasi palsu
- Jika informasi tidak tersedia,
  katakan dengan jujur
- Fokus pada bantuan akademik dan kampus
- Jika pertanyaan di luar topik kampus,
  tolak dengan sopan

Gaya jawaban:
- Jelas
- Ringkas
- Informatif
- Mudah dipahami
- Terstruktur
"""


# =========================================
# OUT OF TOPIC DETECTION
# =========================================

def is_out_of_topic(text):

    text = text.lower()

    campus_keywords = [

        # =================================
        # KAMPUS
        # =================================

        "kampus",
        "universitas",
        "fakultas",
        "jurusan",
        "program studi",

        # =================================
        # AKADEMIK
        # =================================

        "kuliah",
        "mahasiswa",
        "dosen",
        "kurikulum",
        "mata kuliah",

        # =================================
        # ADMINISTRASI
        # =================================

        "ukt",
        "biaya",
        "beasiswa",
        "pendaftaran",
        "akreditasi",

        # =================================
        # FASILITAS
        # =================================

        "laboratorium",
        "perpustakaan",
        "wifi",

        # =================================
        # KARIER
        # =================================

        "karier",
        "prospek kerja",
        "lulusan",

        # =================================
        # PROGRAM STUDI
        # =================================

        "informatika",
        "sistem informasi",
        "data science",
        "cyber security",
        "ai"
    ]

    for keyword in campus_keywords:

        if keyword in text:

            return False

    return True


# =========================================
# OUT OF TOPIC RESPONSE
# =========================================

def out_of_topic_response():

    return """
Maaf, saya hanya dapat membantu terkait informasi kampus dan akademik.

Silakan tanyakan hal seperti:
- Universitas
- Jurusan
- UKT
- Kurikulum
- Beasiswa
- Fasilitas kampus
- Akreditasi
- Prospek karier
"""


# =========================================
# GENERATE RECOMMENDATION
# =========================================

def generate_recommendation(user_id):

    topics = user_memory[user_id]["topics"]

    recommendations = []

    # =====================================
    # PROGRAM STUDI
    # =====================================

    if "informatika" in topics:

        recommendations.append(
            "📚 Program Studi Teknik Informatika"
        )

    if "sistem informasi" in topics:

        recommendations.append(
            "📚 Program Studi Sistem Informasi"
        )

    # =====================================
    # AKADEMIK
    # =====================================

    if "kurikulum" in topics:

        recommendations.append(
            "🧠 Informasi Kurikulum dan Mata Kuliah"
        )

    if "akreditasi" in topics:

        recommendations.append(
            "🌐 Informasi Akreditasi Kampus"
        )

    # =====================================
    # ADMINISTRASI
    # =====================================

    if "biaya" in topics or "ukt" in topics:

        recommendations.append(
            "💰 Informasi UKT dan Pembayaran"
        )

    if "beasiswa" in topics:

        recommendations.append(
            "📖 Informasi Beasiswa Kampus"
        )

    if "pendaftaran" in topics:

        recommendations.append(
            "📝 Informasi Pendaftaran Mahasiswa Baru"
        )

    # =====================================
    # FASILITAS
    # =====================================

    if "laboratorium" in topics:

        recommendations.append(
            "🏢 Informasi Laboratorium Kampus"
        )

    # =====================================
    # KARIER
    # =====================================

    if "karier" in topics:

        recommendations.append(
            "🚀 Prospek Karier Lulusan"
        )

    # =====================================
    # EMPTY RECOMMENDATION
    # =====================================

    if len(recommendations) == 0:

        return ""

    recommendation_text = """

========================================
REKOMENDASI UNTUK ANDA
========================================
"""

    for index, item in enumerate(
        recommendations,
        start=1
    ):

        recommendation_text += (
            f"\n{index}. {item}"
        )

    return recommendation_text


# =========================================
# GENERATE AI RESPONSE
# =========================================

def generate_ai_response(
    user_id,
    user_question
):

    # =====================================
    # INITIALIZE USER
    # =====================================

    initialize_user(user_id)

    # =====================================
    # USER BLOCKED
    # =====================================

    if is_user_blocked(user_id):

        return """
⛔ Akses percakapan dibatasi sementara
karena terdeteksi pelanggaran etika komunikasi.
"""

    # =====================================
    # TOXIC DETECTION
    # =====================================

    toxic_words = detect_toxicity(
        user_question
    )

    if toxic_words:

        return handle_toxicity(
            user_id,
            toxic_words
        )

    # =====================================
    # OUT OF TOPIC DETECTION
    # =====================================

    if is_out_of_topic(user_question):

        return out_of_topic_response()

    # =====================================
    # REPEATED QUESTION DETECTION
    # =====================================

    is_repeat, similarity_score = (
        is_repeated_question(
            user_id,
            user_question
        )
    )

    if is_repeat:

        return repeated_question_response(
            similarity_score
        )

    # =====================================
    # TRACK USER INTEREST
    # =====================================

    track_user_interest(
        user_id,
        user_question
    )

    # =====================================
    # SEARCH KNOWLEDGE
    # =====================================

    knowledge = search_knowledge(
        user_question
    )

    # =====================================
    # BUILD CONTEXT
    # =====================================

    context = build_context(
        knowledge
    )

    # =====================================
    # BUILD FINAL RAG PROMPT
    # =====================================

    final_prompt = f"""
{SYSTEM_PROMPT}

========================================
KNOWLEDGE BASE
========================================

{context}

========================================
PERTANYAAN USER
========================================

{user_question}

========================================
INSTRUKSI MENJAWAB
========================================

- Jawab berdasarkan knowledge base
- Jangan membuat informasi palsu
- Gunakan bahasa Indonesia
- Gunakan format rapi
- Jika ada angka atau biaya,
  tampilkan dengan jelas
- Jika informasi tidak tersedia,
  katakan dengan jujur

========================================
JAWABAN AI
========================================
"""

    # =====================================
    # GENERATE GEMINI RESPONSE
    # =====================================

    try:

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=final_prompt
        )

        ai_answer = response.text

    # =====================================
    # HANDLE API ERROR
    # =====================================

    except Exception as e:

        error_message = str(e)

        # =================================
        # API LIMIT
        # =================================

        if (
            "429" in error_message
            or
            "RESOURCE_EXHAUSTED"
            in error_message
        ):

            ai_answer = """
Permintaan sementara tidak dapat diproses
karena limit penggunaan API.

Silakan coba beberapa saat lagi.
"""

        # =================================
        # API KEY ERROR
        # =================================

        elif "API key" in error_message:

            ai_answer = """
API Key Gemini tidak valid.
"""

        # =================================
        # GENERAL ERROR
        # =================================

        else:

            ai_answer = """
Terjadi kesalahan saat memproses permintaan.
Silakan coba kembali.
"""

    # =====================================
    # SAVE MEMORY
    # =====================================

    save_to_memory(
        user_id,
        user_question
    )

    # =====================================
    # ADD RECOMMENDATION
    # =====================================

    ai_answer += generate_recommendation(
        user_id
    )

    return ai_answer


# =========================================
# TEST AI RESPONSE ENGINE
# =========================================

test_user = "mahasiswa_demo"

question = (
    "berapa biaya kuliah teknik informatika?"
)

answer = generate_ai_response(
    test_user,
    question
)

print("\n🎓 AI RESPONSE\n")

print(answer)


🎓 AI RESPONSE

Mohon maaf, saat ini kami belum dapat memberikan informasi mengenai biaya kuliah program studi Teknik Informatika. Informasi tersebut tidak ditemukan dalam basis data kami.

Untuk mendapatkan informasi terkini dan akurat mengenai biaya kuliah (UKT) program studi Teknik Informatika, kami sarankan Anda untuk:
*   Mengunjungi situs web resmi universitas pada bagian penerimaan mahasiswa baru atau informasi keuangan.
*   Menghubungi langsung bagian Akademik atau Keuangan universitas.

Terima kasih atas pengertian Anda.

REKOMENDASI UNTUK ANDA

1. 📚 Program Studi Teknik Informatika
2. 💰 Informasi UKT dan Pembayaran


# **9. THEME SETTINGS**

In [39]:
import os
from pathlib import Path

# ==================================================
# PILIH THEME DI SINI SAJA
# ==================================================
# "light"
# "dark"
# ==================================================

THEME_MODE = "light"

# ==================================================
# AUTO CREATE .streamlit/config.toml
# ==================================================

os.makedirs(".streamlit", exist_ok=True)

config = f'''
[theme]
base="{THEME_MODE}"

[client]
toolbarMode = "minimal"
showSidebarNavigation = false
'''

with open(".streamlit/config.toml", "w", encoding="utf-8") as f:
    f.write(config)


# ==================================================
# AUTO UPDATE css_loader.py
# ==================================================

if THEME_MODE == "light":
    active_css = "light_mode.css"
else:
    active_css = "dark_mode.css"

css_loader_content = f'''from pathlib import Path

ACTIVE_THEME = "{active_css}"

def load_all_css():
    # Define the folder that contains all CSS files
    css_dir = Path(__file__).parent / "styles"

    # List of CSS files that will be loaded in order
    css_files = [
        "base.css",
        ACTIVE_THEME
    ]

    # Variable to store all combined CSS content
    all_css = ""

    # Loop through each CSS file
    for file in css_files:
        # Create full file path
        path = css_dir / file

        # Check if the file exists before reading it
        if path.exists():
            # Read file content and append it to all_css
            all_css += path.read_text(encoding="utf-8") + "\\n"

    # Return all combined CSS content
    return all_css
'''

loader_path = "css_loader.py"

with open(loader_path, "w", encoding="utf-8") as f:
    f.write(css_loader_content)

print("Theme aktif:", THEME_MODE)

config.toml updated!
css_loader.py updated!
Theme aktif: light


# **10. STREAMLIT RUNTIME CONFIGURATION**



In [19]:
from pyngrok import ngrok
from google.colab import userdata
import subprocess
import time

# Auth token
NGROK_AUTH_TOKEN = userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

proc = None

def run_streamlit(filename, port=8501):

    global proc

    # Stop old processes
    subprocess.run(["pkill", "-f", "streamlit"])
    subprocess.run(["pkill", "-f", "ngrok"])

    # Start Streamlit
    proc = subprocess.Popen(
        [
            "streamlit",
            "run",
            filename,
            "--server.port",
            str(port),
            "--server.headless",
            "true"
        ]
    )

    # Wait for boot
    time.sleep(5)

    # Create ngrok tunnel
    public_url = ngrok.connect(port)

    print("🚀 Streamlit URL:")
    print(public_url)

    return proc


def stop_streamlit():

    global proc

    # Stop Streamlit process
    if proc:
        proc.terminate()
        proc.wait()
        print("✅ Streamlit stopped.")

    # Kill remaining processes
    subprocess.run(["pkill", "-f", "streamlit"])
    subprocess.run(["pkill", "-f", "ngrok"])

    # Close ngrok tunnels
    ngrok.kill()

    print("✅ Ngrok tunnels closed.")

# **11. START APP MANUALLY**

In [20]:
proc = run_streamlit("app.py")

🚀 Streamlit URL:
NgrokTunnel: "https://dazzler-dyslexia-repackage.ngrok-free.dev" -> "http://localhost:8501"


# **12. STOP APP MANUALLY**

In [14]:
# Buka kode dibawah ini untuk stop aplikasi secara manual
# stop_streamlit()